# KoBART Summarizer 학습 (Google Colab)
lotte-insight 프로젝트 — `training/train_summarizer.py` Colab 실행용 노트북

In [ ]:
# 1. GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available')
print('CUDA:', torch.version.cuda)

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 레포 클론 및 의존성 설치
# GitHub URL을 본인 레포로 교체하시오
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/lotte-insight.git'

!git clone {GITHUB_REPO_URL} /content/lotte-insight
%cd /content/lotte-insight/training
!pip install -q -r requirements.txt sentencepiece

In [ ]:
# 4. 데이터 파일을 Google Drive에서 복사
# Drive에 미리 업로드해야 할 파일:
#   MyDrive/lotte-insight-data/labeled_titles.csv           <- data/labeled_titles.csv 그대로
#   MyDrive/lotte-insight-data/labeled_players.resolved.csv <- artifacts/labeled_players.resolved.csv
#
# labeled_players.review_resolutions.csv 은 감사 로그이므로 업로드 불필요
import shutil, os

DRIVE_DATA_DIR = '/content/drive/MyDrive/lotte-insight-data'
LOCAL_DATA_DIR = '/content/lotte-insight/training/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# labeled_titles.csv: 파일명 그대로 복사
src = f'{DRIVE_DATA_DIR}/labeled_titles.csv'
dst = f'{LOCAL_DATA_DIR}/labeled_titles.csv'
if os.path.exists(src):
    shutil.copy(src, dst)
    print('Copied: labeled_titles.csv')
else:
    print(f'[WARN] Not found in Drive: {src}')

# labeled_players.resolved.csv -> data/labeled_players.csv 로 배치
src = f'{DRIVE_DATA_DIR}/labeled_players.resolved.csv'
dst = f'{LOCAL_DATA_DIR}/labeled_players.csv'
if os.path.exists(src):
    shutil.copy(src, dst)
    print('Copied: labeled_players.resolved.csv -> data/labeled_players.csv')
else:
    print(f'[WARN] Not found in Drive: {src}')

In [ ]:
# 5. 데이터 행 수 확인
import pandas as pd

for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{LOCAL_DATA_DIR}/{fname}'
    if os.path.exists(path):
        df = pd.read_csv(path, encoding='utf-8-sig')
        lotte = df[df['is_lotte_related'].astype(str).str.lower() == 'true']
        if 'event_summary' in df.columns:
            with_summary = lotte['event_summary'].fillna('').astype(str).str.strip().ne('').sum()
        else:
            with_summary = 'N/A (event_summary 컬럼 없음 — add_summaries 미실행)'
        print(f'{fname}: total={len(df)}, lotte={len(lotte)}, with_summary={with_summary}')
    else:
        print(f'[MISSING] {fname}')

In [ ]:
# 5-1. train_summarizer.py 핵심 함수 패치 (로컬 변경사항이 push되지 않은 경우 실행)
patch = r'''
import re, pathlib

path = pathlib.Path("train_summarizer.py")
src = path.read_text(encoding="utf-8")

# 1) build_target_text → event_summary 평문만 반환
src = re.sub(
    r"def build_target_text\(row: dict\) -> str:.*?(?=\ndef |\nclass |\Z)",
    (
        "def build_target_text(row: dict) -> str:\n"
        "    return str(row.get(\"event_summary\", \"\") or \"\").strip()\n\n"
    ),
    src,
    flags=re.DOTALL,
)

# 2) compute_metrics → char_f1 기반으로 교체
old_metrics = re.search(
    r"def compute_metrics\(eval_pred\):.*?(?=\ntokenizer_for_metrics)",
    src, re.DOTALL
)
new_metrics = (
    "def _char_f1(pred: str, ref: str) -> float:\n"
    "    pred_chars = set(pred)\n"
    "    ref_chars = set(ref)\n"
    "    if not pred_chars or not ref_chars:\n"
    "        return 0.0\n"
    "    common = pred_chars & ref_chars\n"
    "    precision = len(common) / len(pred_chars)\n"
    "    recall = len(common) / len(ref_chars)\n"
    "    if precision + recall == 0.0:\n"
    "        return 0.0\n"
    "    return 2 * precision * recall / (precision + recall)\n"
    "\n"
    "def compute_metrics(eval_pred):\n"
    "    predictions, labels = eval_pred\n"
    "    if isinstance(predictions, tuple):\n"
    "        predictions = predictions[0]\n"
    "    labels = np.where(labels != -100, labels, 0)\n"
    "    pred_texts = tokenizer_for_metrics.batch_decode(predictions, skip_special_tokens=True)\n"
    "    label_texts = tokenizer_for_metrics.batch_decode(labels, skip_special_tokens=True)\n"
    "    exact_match = 0\n"
    "    char_f1_total = 0.0\n"
    "    for pred_text, label_text in zip(pred_texts, label_texts, strict=False):\n"
    "        pred_text = pred_text.strip()\n"
    "        label_text = label_text.strip()\n"
    "        if pred_text == label_text:\n"
    "            exact_match += 1\n"
    "        char_f1_total += _char_f1(pred_text, label_text)\n"
    "    total = max(len(pred_texts), 1)\n"
    "    return {\n"
    "        \"exact_match\": exact_match / total,\n"
    "        \"char_f1\": char_f1_total / total,\n"
    "    }\n"
    "\n"
)
if old_metrics:
    src = src[:old_metrics.start()] + new_metrics + src[old_metrics.end():]

path.write_text(src, encoding="utf-8")
print("Patch applied. Verifying...")
'''

exec(patch)

# 패치 확인
import subprocess
result = subprocess.run(["python", "-c", "import train_summarizer; print('build_target_text OK')"], capture_output=True, text=True)
print(result.stdout or result.stderr)

In [ ]:
# 6. 학습 실행
# target: event_summary 한국어 텍스트 단독 (JSON 아님)
# 지표: eval_loss (best model 선택 기준), char_f1 (문자 F1), exact_match
!python train_summarizer.py \
    --epochs 5 \
    --batch 8 \
    --max-source-len 256 \
    --max-target-len 192 \
    --num-beams 4 \
    --early-stopping-patience 2

In [ ]:
# 7. 학습된 모델을 Google Drive에 저장
DRIVE_MODEL_DIR = '/content/drive/MyDrive/lotte-insight-data/models/summarizer_kobart'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

LOCAL_MODEL_DIR = '/content/lotte-insight/training/models/summarizer_kobart'
if os.path.exists(LOCAL_MODEL_DIR):
    shutil.copytree(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR, dirs_exist_ok=True)
    print(f'Model saved to Drive: {DRIVE_MODEL_DIR}')
    print('Files:', os.listdir(DRIVE_MODEL_DIR))
else:
    print('[ERROR] Model directory not found — training may have failed')

In [ ]:
# 8. (선택) 평가만 실행
# !python train_summarizer.py --eval-only